# Faster R-CNN dataset EDA

Quick look at the COCO-formatted custom dataset before training:

- class distribution
- image size histogram
- bbox area / aspect ratio distribution
- a few sample images with their boxes drawn

In [ ]:
import json, os
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
from src.visualize import draw_boxes

ROOT = 'data/voc'
ANN  = os.path.join(ROOT, 'annotations', 'train.json')
with open(ANN) as f:
    coco = json.load(f)
print('images:', len(coco['images']), 'anns:', len(coco['annotations']))

In [ ]:
id2name = {c['id']: c['name'] for c in coco['categories']}
counts = Counter([id2name[a['category_id']] for a in coco['annotations']])
names, vals = zip(*counts.most_common())
plt.figure(figsize=(10,4))
plt.bar(names, vals)
plt.xticks(rotation=45, ha='right')
plt.title('class distribution')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
ws = np.array([im['width'] for im in coco['images']])
hs = np.array([im['height'] for im in coco['images']])
fig, ax = plt.subplots(1,2, figsize=(10,3))
ax[0].hist(ws, bins=30); ax[0].set_title('image width')
ax[1].hist(hs, bins=30); ax[1].set_title('image height')
plt.tight_layout(); plt.show()

In [ ]:
import random
from collections import defaultdict
by_im = defaultdict(list)
for a in coco['annotations']:
    by_im[a['image_id']].append(a)
samples = random.sample(list(by_im.keys()), 4)
fig, axes = plt.subplots(2,2, figsize=(10,10))
for ax, im_id in zip(axes.flat, samples):
    im_meta = next(m for m in coco['images'] if m['id'] == im_id)
    img = Image.open(os.path.join(ROOT, im_meta['file_name'])).convert('RGB')
    boxes  = [[a['bbox'][0], a['bbox'][1], a['bbox'][0]+a['bbox'][2], a['bbox'][1]+a['bbox'][3]] for a in by_im[im_id]]
    labels = [a['category_id'] for a in by_im[im_id]]
    scores = [1.0]*len(boxes)
    drawn  = draw_boxes(img, boxes, labels, scores)
    ax.imshow(drawn); ax.axis('off')
plt.tight_layout(); plt.show()